## Experiment No: 10
## Experiment Title: Mini Project - AI-Powered News Article Summarizer

**Name:** Himanshu Jadhav  
**Roll Number:** TE-33

### Step 1: Import Libraries

In [1]:
import re
import numpy as np
import pandas as pd
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from rouge_score import rouge_scorer
from tabulate import tabulate

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

### Step 2: Sample News Articles Dataset

In [2]:
articles = [
    {
        "title": "Tech Company Launches New AI Model",
        "text": """A leading technology company announced the release of its newest
        artificial intelligence model this week. The model is designed to improve
        natural language understanding and can generate human-like text responses.
        Company executives said the new system outperforms previous versions on
        several benchmark tests. The model will be available to developers through
        an API starting next month. Analysts believe this launch could intensify
        competition among major AI companies. Early testers have praised the model
        for its speed and accuracy in handling complex queries.""",
        "reference": "A tech company launched a new AI model that outperforms previous versions and will be available via API next month."
    },
    {
        "title": "City Announces New Public Transport Plan",
        "text": """The city council approved a new public transportation plan aimed at
        reducing traffic congestion. The plan includes the construction of two new
        metro lines and expansion of the existing bus network. Officials said the
        project will take five years to complete and cost several million dollars.
        Residents have expressed mixed reactions, with some welcoming the improved
        connectivity and others concerned about construction disruptions. The first
        phase of construction is expected to begin early next year.""",
        "reference": "The city approved a five-year public transport plan with new metro lines and expanded buses to reduce traffic congestion."
    }
]

for a in articles:
    print("Title:", a["title"])
    print(a["text"][:120], "...")
    print()

Title: Tech Company Launches New AI Model
A leading technology company announced the release of its newest
        artificial intelligence model this week. The mo ...

Title: City Announces New Public Transport Plan
The city council approved a new public transportation plan aimed at
        reducing traffic congestion. The plan includ ...



### Step 3: Text Cleaning and Preprocessing

In [3]:
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)   # collapse extra whitespace
    return text.strip()

for a in articles:
    a["text"] = clean_text(a["text"])

print("Cleaned Article 1:")
print(articles[0]["text"])

Cleaned Article 1:
A leading technology company announced the release of its newest artificial intelligence model this week. The model is designed to improve natural language understanding and can generate human-like text responses. Company executives said the new system outperforms previous versions on several benchmark tests. The model will be available to developers through an API starting next month. Analysts believe this launch could intensify competition among major AI companies. Early testers have praised the model for its speed and accuracy in handling complex queries.


### Step 4: Extractive Summarization Function (TF-IDF based)

In [4]:
def extractive_summary(text, num_sentences=2):
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text

    vectorizer = TfidfVectorizer(stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sentences)
    scores = tfidf_matrix.sum(axis=1).A1

    top_indices = sorted(np.argsort(scores)[-num_sentences:])
    return ' '.join([sentences[i] for i in top_indices])

for a in articles:
    a['extractive_summary'] = extractive_summary(a['text'])

print('Extractive Summary (Article 1):')
print(articles[0]['extractive_summary'])

Extractive Summary (Article 1):
A leading technology company announced the release of its newest artificial intelligence model this week. The model is designed to improve natural language understanding and can generate human-like text responses.


### Step 5: Load Abstractive Summarization Model

In [5]:
MODEL_NAME = 'sshleifer/distilbart-cnn-12-6'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print('Model loaded successfully:', MODEL_NAME)

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded successfully: sshleifer/distilbart-cnn-12-6


### Step 6: Generate Abstractive Summaries

In [6]:
for a in articles:
    inputs = tokenizer(a['text'], return_tensors='pt', truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_length=45,
        min_length=15,
        num_beams=4,
        do_sample=False,
        forced_bos_token_id=0
    )
    a['abstractive_summary'] = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

for a in articles:
    print('Title:', a['title'])
    print('Abstractive Summary:', a['abstractive_summary'])
    print()

Title: Tech Company Launches New AI Model
Abstractive Summary:  Company executives said the new system outperforms previous versions on several benchmark tests . The model will be available to developers through an API starting next month . Analysts believe this launch could intensify competition among major AI companies .

Title: City Announces New Public Transport Plan
Abstractive Summary:  The city council approved a new public transportation plan aimed at reducing traffic congestion . The plan includes the construction of two new metro lines and expansion of the existing bus network . Officials said the project will take five years to



### Step 7: Evaluate Both Summaries using ROUGE

In [7]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

results = []
for a in articles:
    ext_scores = scorer.score(a["reference"], a["extractive_summary"])
    abs_scores = scorer.score(a["reference"], a["abstractive_summary"])

    results.append({
        "Article": a["title"],
        "Method": "Extractive",
        "ROUGE-1": round(ext_scores["rouge1"].fmeasure, 3),
        "ROUGE-2": round(ext_scores["rouge2"].fmeasure, 3),
        "ROUGE-L": round(ext_scores["rougeL"].fmeasure, 3)
    })
    results.append({
        "Article": a["title"],
        "Method": "Abstractive",
        "ROUGE-1": round(abs_scores["rouge1"].fmeasure, 3),
        "ROUGE-2": round(abs_scores["rouge2"].fmeasure, 3),
        "ROUGE-L": round(abs_scores["rougeL"].fmeasure, 3)
    })

results_df = pd.DataFrame(results)
print(tabulate(results_df, showindex=False, headers='keys', tablefmt='fancy_grid'))

╒══════════════════════════════════════════╤═════════════╤═══════════╤═══════════╤═══════════╕
│ Article                                  │ Method      │   ROUGE-1 │   ROUGE-2 │   ROUGE-L │
╞══════════════════════════════════════════╪═════════════╪═══════════╪═══════════╪═══════════╡
│ Tech Company Launches New AI Model       │ Extractive  │     0.157 │     0     │     0.157 │
├──────────────────────────────────────────┼─────────────┼───────────┼───────────┼───────────┤
│ Tech Company Launches New AI Model       │ Abstractive │     0.491 │     0.182 │     0.386 │
├──────────────────────────────────────────┼─────────────┼───────────┼───────────┼───────────┤
│ City Announces New Public Transport Plan │ Extractive  │     0.51  │     0.245 │     0.392 │
├──────────────────────────────────────────┼─────────────┼───────────┼───────────┼───────────┤
│ City Announces New Public Transport Plan │ Abstractive │     0.567 │     0.345 │     0.4   │
╘══════════════════════════════════════════╧══════

### Step 8: Display Final Results Neatly

In [8]:
for a in articles:
    print("=" * 70)
    print("TITLE:", a["title"])
    print("-" * 70)
    print("Reference Summary   :", a["reference"])
    print("Extractive Summary  :", a["extractive_summary"])
    print("Abstractive Summary :", a["abstractive_summary"])
    print()

TITLE: Tech Company Launches New AI Model
----------------------------------------------------------------------
Reference Summary   : A tech company launched a new AI model that outperforms previous versions and will be available via API next month.
Extractive Summary  : A leading technology company announced the release of its newest artificial intelligence model this week. The model is designed to improve natural language understanding and can generate human-like text responses.
Abstractive Summary :  Company executives said the new system outperforms previous versions on several benchmark tests . The model will be available to developers through an API starting next month . Analysts believe this launch could intensify competition among major AI companies .

TITLE: City Announces New Public Transport Plan
----------------------------------------------------------------------
Reference Summary   : The city approved a five-year public transport plan with new metro lines and expanded b

### Final Output

In [9]:
print("Experiment Completed Successfully")
print()
print(tabulate(results_df, showindex=False, headers='keys', tablefmt='fancy_grid'))

Experiment Completed Successfully

╒══════════════════════════════════════════╤═════════════╤═══════════╤═══════════╤═══════════╕
│ Article                                  │ Method      │   ROUGE-1 │   ROUGE-2 │   ROUGE-L │
╞══════════════════════════════════════════╪═════════════╪═══════════╪═══════════╪═══════════╡
│ Tech Company Launches New AI Model       │ Extractive  │     0.157 │     0     │     0.157 │
├──────────────────────────────────────────┼─────────────┼───────────┼───────────┼───────────┤
│ Tech Company Launches New AI Model       │ Abstractive │     0.491 │     0.182 │     0.386 │
├──────────────────────────────────────────┼─────────────┼───────────┼───────────┼───────────┤
│ City Announces New Public Transport Plan │ Extractive  │     0.51  │     0.245 │     0.392 │
├──────────────────────────────────────────┼─────────────┼───────────┼───────────┼───────────┤
│ City Announces New Public Transport Plan │ Abstractive │     0.567 │     0.345 │     0.4   │
╘══════════════